# File Naming Rule Migration

旧命名規則を新命名規則へ一括変換する Notebook です。

- Program A (canonical)
  - old: `ERA5.STAGE1.<Var>.<YYYY>.nc[.tmp...]`
  - new: `ERA5.<Var>.<YYYY>.ILS.nc[.tmp...]`
- Program B (30min)
  - old: `GSWP3.BC.<Var>.1hrMap.ILS.<YYYY>.nc[.tmp...]`
  - new: `ERA5.30min.<Var>.1hrMap.ILS.<YYYY>.nc[.tmp...]`

実行手順: 上から順に実行し、最後のセルで `DRY_RUN=False` にして実行してください。

In [2]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple
import re


# 必要なら絶対パスを指定（通常は空文字のままでOK）
PROJECT_ROOT_OVERRIDE = ""


def find_project_root(explicit: Optional[str] = None) -> Path:
    if explicit:
        p = Path(explicit).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {p}")
        return p

    start = Path.cwd().resolve()
    for cand in [start] + list(start.parents):
        if (cand / "configs" / "convert_default.yaml").exists() and (cand / "scripts").exists():
            return cand
    return start


PROJECT_ROOT = find_project_root(PROJECT_ROOT_OVERRIDE or None)
SEARCH_ROOTS = [
    PROJECT_ROOT / "era5_canonical",
    PROJECT_ROOT / "30min",
]

print(f"cwd         : {Path.cwd()}")
print(f"project_root: {PROJECT_ROOT}")
for r in SEARCH_ROOTS:
    print(f"search_root : {r} (exists={r.exists()})")

RE_STAGE1 = re.compile(
    r"^ERA5\.STAGE1\.(?P<var>[A-Za-z0-9_]+)\.(?P<year>\d{4})\.nc(?P<tmp>\.tmp(?:\.\d+)?)?$"
)
RE_30MIN = re.compile(
    r"^GSWP3\.BC\.(?P<var>[A-Za-z0-9_]+)\.1hrMap\.ILS\.(?P<year>\d{4})\.nc(?P<tmp>\.tmp(?:\.\d+)?)?$"
)


@dataclass
class RenameItem:
    src: Path
    dst: Path
    category: str


def map_new_name(name: str) -> Optional[Tuple[str, str]]:
    m1 = RE_STAGE1.match(name)
    if m1:
        tmp = m1.group("tmp") or ""
        return (f"ERA5.{m1.group('var')}.{m1.group('year')}.ILS.nc{tmp}", "canonical")

    m2 = RE_30MIN.match(name)
    if m2:
        tmp = m2.group("tmp") or ""
        return (f"ERA5.30min.{m2.group('var')}.1hrMap.ILS.{m2.group('year')}.nc{tmp}", "30min")

    return None


def collect_candidates(search_roots: List[Path]) -> List[RenameItem]:
    items: List[RenameItem] = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            mapped = map_new_name(p.name)
            if mapped is None:
                continue
            new_name, category = mapped
            dst = p.with_name(new_name)
            if dst != p:
                items.append(RenameItem(src=p, dst=dst, category=category))
    items.sort(key=lambda x: str(x.src))
    return items


candidates = collect_candidates(SEARCH_ROOTS)
print(f"Found candidates: {len(candidates)}")
print(f" - canonical: {sum(1 for x in candidates if x.category == 'canonical')}")
print(f" - 30min:     {sum(1 for x in candidates if x.category == '30min')}")

if not any(r.exists() for r in SEARCH_ROOTS):
    print("WARNING: search roots not found. PROJECT_ROOT_OVERRIDE を設定してください。")

cwd         : /data02/shiojiri/ILS/ILS_data/ERA5/notebooks
project_root: /data02/shiojiri/ILS/ILS_data/ERA5
search_root : /data02/shiojiri/ILS/ILS_data/ERA5/era5_canonical (exists=True)
search_root : /data02/shiojiri/ILS/ILS_data/ERA5/30min (exists=True)
Found candidates: 182
 - canonical: 99
 - 30min:     83


In [3]:
PREVIEW_N = 50
for i, item in enumerate(candidates[:PREVIEW_N], start=1):
    print(f"[{i:04d}] {item.src} -> {item.dst}")
if len(candidates) > PREVIEW_N:
    print(f"... ({len(candidates) - PREVIEW_N} more)")

[0001] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.CCover.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.CCover.1hrMap.ILS.2013.nc
[0002] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.LWdown.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.LWdown.1hrMap.ILS.2013.nc
[0003] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.PSurf.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.PSurf.1hrMap.ILS.2013.nc
[0004] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.Qair.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.Qair.1hrMap.ILS.2013.nc
[0005] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.Rainf.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.Rainf.1hrMap.ILS.2013.nc
[0006] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.SWdown.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/E

In [4]:
# DRY_RUN=True: 変更しない（表示のみ）
# DRY_RUN=False: 実際に rename 実行
DRY_RUN = False

renamed = 0
skipped_exists = 0
errors = 0

for item in candidates:
    if item.dst.exists() and item.dst != item.src:
        skipped_exists += 1
        print(f"[SKIP exists] {item.src} -> {item.dst}")
        continue

    if DRY_RUN:
        print(f"[DRY] {item.src} -> {item.dst}")
        continue

    try:
        item.src.rename(item.dst)
        renamed += 1
        print(f"[OK] {item.src} -> {item.dst}")
    except Exception as e:
        errors += 1
        print(f"[ERR] {item.src} -> {item.dst}: {e}")

print("---")
print(f"dry_run={DRY_RUN}")
print(f"renamed={renamed}")
print(f"skipped_exists={skipped_exists}")
print(f"errors={errors}")

[OK] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.CCover.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.CCover.1hrMap.ILS.2013.nc
[OK] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.LWdown.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.LWdown.1hrMap.ILS.2013.nc
[OK] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.PSurf.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.PSurf.1hrMap.ILS.2013.nc
[OK] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.Qair.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.Qair.1hrMap.ILS.2013.nc
[OK] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.Rainf.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.Rainf.1hrMap.ILS.2013.nc
[OK] /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/GSWP3.BC.SWdown.1hrMap.ILS.2013.nc -> /data02/shiojiri/ILS/ILS_data/ERA5/30min/2013/ERA5.30min.SW